In [ ]:
# =========================
# DATASET ANALYSIS CONFIG
# =========================
import os
import json
import math
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_distances

import torch
import torch.nn.functional as F
from torchvision import models, transforms

# ====== CONFIG ======
# Kaggle: .../foodseg103-fullmask/foodseg103-full
# Local: set FOODSEG103_ROOT to source/dataset/foodseg103-full
ANALYSIS_CONFIG = {
    "DATA_ROOT": os.environ.get(
        "FOODSEG103_ROOT",
        "/kaggle/input/datasets/arischii05/foodseg103-fullmask/foodseg103-full",
    ),
    "OUTPUT_DIR": os.environ.get("FOODSEG103_OUT", "/kaggle/working/"),
    "SPLIT_JSON_PATH": os.environ.get(
        "FOODSEG103_SPLIT_CACHE",
        "/kaggle/working/split_foodseg103_full.json",
    ),
    "MAX_VIS_SAMPLES": 24,
    "MAX_EMBED_SAMPLES": 800,
    "SEED": 42,
}


In [ ]:
# =========================
# PATHS / SPLIT / MAPPING
# =========================
random.seed(ANALYSIS_CONFIG["SEED"])
np.random.seed(ANALYSIS_CONFIG["SEED"])

DATA_ROOT = Path(ANALYSIS_CONFIG["DATA_ROOT"])
OUT_DIR = Path(ANALYSIS_CONFIG["OUTPUT_DIR"])
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_ROOT / "class_mapping.json", "r", encoding="utf-8") as f:
    class_mapping = json.load(f)

BACKGROUND_ID = int(class_mapping["background_id"])
if "num_classes" in class_mapping:
    NUM_CLASSES = int(class_mapping["num_classes"])
else:
    # foodseg103-full: 103 ingredient ids (0..102) + background_id (103) => 104 labels
    NUM_CLASSES = int(class_mapping["num_ingredient_classes"]) + 1

ID_TO_CLASS = {int(k): v for k, v in class_mapping["id_to_class"].items()}

IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
split_file_path = Path(ANALYSIS_CONFIG["SPLIT_JSON_PATH"])

# Official layout: DATA_ROOT / {train,test} / {img, mask, ann, meta}
SPLIT_NAMES = ["train", "test"]

if not split_file_path.is_file():
    print(f"Không tìm thấy {split_file_path}. Scan thư mục {SPLIT_NAMES}...")

    split_data_ram = {name: [] for name in SPLIT_NAMES}

    for split_name in SPLIT_NAMES:
        img_dir = DATA_ROOT / split_name / "img"
        if img_dir.exists() and img_dir.is_dir():
            stems = [
                f.stem
                for f in img_dir.iterdir()
                if f.is_file() and f.suffix.lower() in IMG_EXTS
            ]
            split_data_ram[split_name] = sorted(stems)

    with open(split_file_path, "w", encoding="utf-8") as f:
        json.dump(split_data_ram, f, indent=4)

    print(f"Đã lưu cache split tại: {split_file_path}")
    split_data = split_data_ram

else:
    with open(split_file_path, "r", encoding="utf-8") as f:
        split_data = json.load(f)

train_stems = split_data.get("train", [])
test_stems = split_data.get("test", split_data.get("val", []))


def find_image_path(img_dir: Path, stem: str):
    """Resolve image file path for a sample stem inside ``img_dir``.

    Args:
        img_dir: Directory containing RGB images.
        stem: Filename stem shared with the paired mask (``{stem}.png``).

    Returns:
        Path to the image file if found, otherwise None.

    Raises:
        None: Missing files return None instead of raising.
    """
    for ext in IMG_EXTS:
        p = img_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None


def load_rgb(path):
    """Load an image as an RGB uint8 array.

    Args:
        path: Filesystem path to an image file.

    Returns:
        ``numpy.ndarray`` of shape (H, W, 3), dtype uint8.

    Raises:
        None: Delegates PIL errors to the caller.
    """
    return np.array(Image.open(path).convert("RGB"))


def load_mask(path):
    """Load a single-channel segmentation mask as uint8.

    Args:
        path: Filesystem path to a mask image (PNG).

    Returns:
        ``numpy.ndarray`` of shape (H, W), dtype uint8.

    Raises:
        None: Delegates PIL errors to the caller.
    """
    return np.array(Image.open(path), dtype=np.uint8)


print("NUM_CLASSES:", NUM_CLASSES)
print("BACKGROUND_ID:", BACKGROUND_ID)
print("Số lượng - train:", len(train_stems), "| test:", len(test_stems))
print("OUT_DIR:", OUT_DIR)


In [ ]:
# =========================
# BUILD SAMPLE METADATA
# =========================
def analyze_sample(split: str, stem: str):
    """Compute per-image statistics for one FoodSeg103 sample.

    Args:
        split: Split name (``train`` or ``test``).
        stem: Image/mask stem (e.g. ``00000000``).

    Returns:
        Dict row suitable for ``pd.DataFrame``, or None if files are missing.

    Raises:
        None: Invalid/missing pairs return None.
    """
    img_dir = DATA_ROOT / split / "img"
    mask_dir = DATA_ROOT / split / "mask"

    img_path = find_image_path(img_dir, stem)
    mask_path = mask_dir / f"{stem}.png"

    if img_path is None or not mask_path.exists():
        return None

    image = load_rgb(img_path)
    mask = load_mask(mask_path)

    h, w = mask.shape
    uniq, counts = np.unique(mask, return_counts=True)
    pix_dict = {int(k): int(v) for k, v in zip(uniq, counts)}

    fg_pixels = int(sum(v for k, v in pix_dict.items() if k != BACKGROUND_ID))
    bg_pixels = int(pix_dict.get(BACKGROUND_ID, 0))
    total_pixels = int(mask.size)
    fg_ratio = fg_pixels / max(1, total_pixels)
    bg_ratio = bg_pixels / max(1, total_pixels)

    present_classes = sorted([int(k) for k in pix_dict.keys() if int(k) != BACKGROUND_ID])
    num_present_classes = len(present_classes)

    fg_binary = (mask != BACKGROUND_ID).astype(np.uint8)
    num_cc, cc_map, stats, _ = cv2.connectedComponentsWithStats(fg_binary, connectivity=8)
    cc_areas = stats[1:, cv2.CC_STAT_AREA] if num_cc > 1 else np.array([], dtype=np.int32)

    row = {
        "split": split,
        "stem": stem,
        "image_path": str(img_path),
        "mask_path": str(mask_path),
        "height": h,
        "width": w,
        "aspect_ratio": w / max(1, h),
        "total_pixels": total_pixels,
        "fg_pixels": fg_pixels,
        "bg_pixels": bg_pixels,
        "fg_ratio": fg_ratio,
        "bg_ratio": bg_ratio,
        "num_present_classes": num_present_classes,
        "present_classes": present_classes,
        "num_connected_components": int(len(cc_areas)),
        "largest_component_area": int(cc_areas.max()) if len(cc_areas) else 0,
        "median_component_area": float(np.median(cc_areas)) if len(cc_areas) else 0.0,
    }

    for cid in range(NUM_CLASSES):
        row[f"class_{cid}_pixels"] = int(pix_dict.get(cid, 0))

    return row


def build_metadata(split: str, stems):
    """Build a metadata table for all stems in a split.

    Args:
        split: Split name passed to ``analyze_sample``.
        stems: Iterable of filename stems.

    Returns:
        ``pandas.DataFrame`` of per-image stats (possibly empty).

    Raises:
        None.
    """
    rows = []
    for stem in tqdm(stems, desc=f"Analyze {split}"):
        row = analyze_sample(split, stem)
        if row is not None:
            rows.append(row)
    return pd.DataFrame(rows)


train_meta = build_metadata("train", train_stems)
test_meta = build_metadata("test", test_stems)
meta_df = pd.concat([train_meta, test_meta], ignore_index=True)

meta_csv = OUT_DIR / "sample_metadata.csv"
meta_df.to_csv(meta_csv, index=False)

print(meta_df.shape)
print("Saved:", meta_csv)
meta_df.head()


In [ ]:
# =========================
# GLOBAL SUMMARY
# =========================
summary = {
    "num_train_samples": int((meta_df["split"] == "train").sum()),
    "num_test_samples": int((meta_df["split"] == "test").sum()),
    "mean_height": float(meta_df["height"].mean()),
    "mean_width": float(meta_df["width"].mean()),
    "median_height": float(meta_df["height"].median()),
    "median_width": float(meta_df["width"].median()),
    "mean_fg_ratio": float(meta_df["fg_ratio"].mean()),
    "median_fg_ratio": float(meta_df["fg_ratio"].median()),
    "mean_num_present_classes": float(meta_df["num_present_classes"].mean()),
    "median_num_present_classes": float(meta_df["num_present_classes"].median()),
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(OUT_DIR / "global_summary.csv", index=False)
summary_df


In [ ]:
# =========================
# CLASS DISTRIBUTION
# =========================
class_rows = []
for cid in range(NUM_CLASSES):
    cname = "background" if cid == BACKGROUND_ID else ID_TO_CLASS.get(cid, f"class_{cid}")
    pixel_col = f"class_{cid}_pixels"

    total_pixels = int(meta_df[pixel_col].sum())
    presence_count = int((meta_df[pixel_col] > 0).sum())
    presence_ratio = presence_count / max(1, len(meta_df))
    mean_pixels_per_image = float(meta_df[pixel_col].mean())
    mean_pixels_when_present = float(meta_df.loc[meta_df[pixel_col] > 0, pixel_col].mean()) if presence_count > 0 else 0.0

    class_rows.append({
        "class_id": cid,
        "class_name": cname,
        "total_pixels": total_pixels,
        "pixel_ratio": total_pixels / max(1, int(meta_df["total_pixels"].sum())),
        "presence_count": presence_count,
        "presence_ratio": presence_ratio,
        "mean_pixels_per_image": mean_pixels_per_image,
        "mean_pixels_when_present": mean_pixels_when_present,
    })

class_df = pd.DataFrame(class_rows).sort_values("total_pixels", ascending=False)
class_df.to_csv(OUT_DIR / "class_distribution.csv", index=False)

print(class_df.head(15))
print("\nRarest classes by pixel:")
print(class_df.sort_values("total_pixels", ascending=True).head(15))

In [ ]:
# =========================
# PLOTS: CLASS IMBALANCE
# =========================
plt.figure(figsize=(16, 5))
plt.bar(class_df["class_id"].astype(str), class_df["pixel_ratio"].values)
plt.xticks(rotation=90)
plt.title("Pixel ratio by class")
plt.tight_layout()
plt.show()

plt.figure(figsize=(16, 5))
plt.bar(class_df["class_id"].astype(str), class_df["presence_ratio"].values)
plt.xticks(rotation=90)
plt.title("Image presence ratio by class")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(meta_df["fg_ratio"], bins=40)
plt.title("Foreground ratio distribution")
plt.xlabel("fg_ratio")
plt.ylabel("count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(meta_df["num_present_classes"], bins=30)
plt.title("Number of classes per image")
plt.xlabel("num_present_classes")
plt.ylabel("count")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# OBJECT SIZE ANALYSIS
# =========================
object_rows = []

for _, row in tqdm(meta_df.iterrows(), total=len(meta_df), desc="Object size analysis"):
    mask = load_mask(row["mask_path"])
    h, w = mask.shape
    total = h * w

    for cid in np.unique(mask):
        cid = int(cid)
        if cid == BACKGROUND_ID:
            continue

        binary = (mask == cid).astype(np.uint8)
        num_cc, cc_map, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)

        for i in range(1, num_cc):
            area = int(stats[i, cv2.CC_STAT_AREA])
            x = int(stats[i, cv2.CC_STAT_LEFT])
            y = int(stats[i, cv2.CC_STAT_TOP])
            bw = int(stats[i, cv2.CC_STAT_WIDTH])
            bh = int(stats[i, cv2.CC_STAT_HEIGHT])

            object_rows.append({
                "stem": row["stem"],
                "class_id": cid,
                "class_name": ID_TO_CLASS.get(cid, f"class_{cid}"),
                "area_pixels": area,
                "area_ratio": area / max(1, total),
                "bbox_w": bw,
                "bbox_h": bh,
                "bbox_area": bw * bh,
                "bbox_area_ratio": (bw * bh) / max(1, total),
            })

object_df = pd.DataFrame(object_rows)
object_df.to_csv(OUT_DIR / "object_components.csv", index=False)

print(object_df.shape)
print(object_df.head())

In [ ]:
# =========================
# OBJECT SUMMARY BY CLASS
# =========================
obj_summary = object_df.groupby(["class_id", "class_name"]).agg(
    num_objects=("area_pixels", "count"),
    mean_area_pixels=("area_pixels", "mean"),
    median_area_pixels=("area_pixels", "median"),
    mean_area_ratio=("area_ratio", "mean"),
    median_area_ratio=("area_ratio", "median"),
    mean_bbox_area_ratio=("bbox_area_ratio", "mean"),
).reset_index()

obj_summary = obj_summary.sort_values("median_area_ratio", ascending=True)
obj_summary.to_csv(OUT_DIR / "object_size_summary_by_class.csv", index=False)

print("Smallest objects by median area ratio:")
print(obj_summary.head(15))

print("\nLargest objects by median area ratio:")
print(obj_summary.tail(15))

In [ ]:
# =========================
# VISUALIZE RANDOM SAMPLES
# =========================
def colorize_mask(mask, num_classes=NUM_CLASSES):
    """Map class indices to random RGB colors for visualization.

    Args:
        mask: Label map (H, W) with integer class ids.
        num_classes: Palette size (must exceed max label id used).

    Returns:
        RGB ``uint8`` array of shape (H, W, 3).

    Raises:
        IndexError: If ``mask`` contains values outside ``0..num_classes-1``.
    """
    rng = np.random.default_rng(123)
    palette = rng.integers(0, 255, size=(num_classes, 3), dtype=np.uint8)
    palette[BACKGROUND_ID] = np.array([0, 0, 0], dtype=np.uint8)
    return palette[mask]


def overlay_mask(image, mask, alpha=0.45):
    """Alpha-blend colored mask over image on foreground pixels.

    Args:
        image: RGB uint8 array (H, W, 3).
        mask: Label map (H, W).
        alpha: Opacity of the overlay on foreground.

    Returns:
        Blended RGB uint8 array.

    Raises:
        None.
    """
    color_mask = colorize_mask(mask)
    over = image.copy()
    fg = (mask != BACKGROUND_ID)[..., None]
    over = np.where(fg, (1 - alpha) * image + alpha * color_mask, image).astype(np.uint8)
    return over


samples = meta_df.sample(min(ANALYSIS_CONFIG["MAX_VIS_SAMPLES"], len(meta_df)), random_state=42)

n = len(samples)
cols = 3
rows = math.ceil(n / cols)

plt.figure(figsize=(16, rows * 4))
for i, (_, row) in enumerate(samples.iterrows(), 1):
    image = load_rgb(row["image_path"])
    mask = load_mask(row["mask_path"])
    over = overlay_mask(image, mask)

    plt.subplot(rows, cols, i)
    plt.imshow(over)
    plt.title(f"{row['stem']}\nfg={row['fg_ratio']:.2f}, cls={row['num_present_classes']}")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# =========================
# HARD / ODD SAMPLES
# =========================
hard_df = meta_df.copy()

# heuristic score: ít foreground + nhiều component nhỏ + quá nhiều class
hard_df["hard_score"] = (
    (1 - hard_df["fg_ratio"]) * 2.0
    + (hard_df["num_connected_components"].clip(upper=50) / 10.0)
    + (hard_df["num_present_classes"] / 5.0)
)

hardest_samples = hard_df.sort_values("hard_score", ascending=False).head(50)
easiest_samples = hard_df.sort_values("hard_score", ascending=True).head(50)

hardest_samples.to_csv(OUT_DIR / "hardest_samples.csv", index=False)
easiest_samples.to_csv(OUT_DIR / "easiest_samples.csv", index=False)

print("Saved hardest/easiest sample lists.")
hardest_samples[["stem", "fg_ratio", "num_connected_components", "num_present_classes", "hard_score"]].head(20)

In [ ]:
# =========================
# EMBEDDING + PCA
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embed_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
embed_model.fc = torch.nn.Identity()
embed_model = embed_model.to(device).eval()

embed_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

embed_df = meta_df.sample(min(ANALYSIS_CONFIG["MAX_EMBED_SAMPLES"], len(meta_df)), random_state=42).reset_index(drop=True)

embeddings = []
with torch.no_grad():
    for _, row in tqdm(embed_df.iterrows(), total=len(embed_df), desc="Extract embeddings"):
        img = load_rgb(row["image_path"])
        x = embed_tf(img).unsqueeze(0).to(device)
        feat = embed_model(x).squeeze(0).cpu().numpy()
        embeddings.append(feat)

embeddings = np.stack(embeddings, axis=0)
print("embeddings:", embeddings.shape)

scaler = StandardScaler()
emb_std = scaler.fit_transform(embeddings)

pca = PCA(n_components=2, random_state=42)
emb_pca = pca.fit_transform(emb_std)

embed_df["pca_1"] = emb_pca[:, 0]
embed_df["pca_2"] = emb_pca[:, 1]

embed_df.to_csv(OUT_DIR / "embedding_pca.csv", index=False)

print("Explained variance ratio:", pca.explained_variance_ratio_)

In [ ]:
# =========================
# PCA VISUALIZATION
# =========================
plt.figure(figsize=(7, 6))
sc = plt.scatter(embed_df["pca_1"], embed_df["pca_2"], c=embed_df["fg_ratio"], s=18)
plt.colorbar(sc, label="fg_ratio")
plt.title("PCA of image embeddings colored by fg_ratio")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 6))
sc = plt.scatter(embed_df["pca_1"], embed_df["pca_2"], c=embed_df["num_present_classes"], s=18)
plt.colorbar(sc, label="num_present_classes")
plt.title("PCA colored by num_present_classes")
plt.tight_layout()
plt.show()

kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
clusters = kmeans.fit_predict(emb_std)
embed_df["cluster"] = clusters

plt.figure(figsize=(7, 6))
plt.scatter(embed_df["pca_1"], embed_df["pca_2"], c=embed_df["cluster"], s=18)
plt.title("PCA colored by KMeans cluster")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# OUTLIER DETECTION
# =========================
center = emb_std.mean(axis=0, keepdims=True)
dists = np.linalg.norm(emb_std - center, axis=1)

embed_df["embed_dist_to_center"] = dists
outliers = embed_df.sort_values("embed_dist_to_center", ascending=False).head(50)
outliers.to_csv(OUT_DIR / "embedding_outliers.csv", index=False)

outliers[["stem", "fg_ratio", "num_present_classes", "embed_dist_to_center"]].head(20)

In [ ]:
# =========================
# VISUALIZE OUTLIERS
# =========================
top_outliers = outliers.head(min(16, len(outliers)))

plt.figure(figsize=(16, 16))
for i, (_, row) in enumerate(top_outliers.iterrows(), 1):
    image = load_rgb(row["image_path"])
    mask = load_mask(row["mask_path"])
    over = overlay_mask(image, mask)

    plt.subplot(4, 4, i)
    plt.imshow(over)
    plt.title(f"{row['stem']}\nd={row['embed_dist_to_center']:.2f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# FOREGROUND-AWARE CROP HELPER
# =========================
def random_foreground_crop(image, mask, crop_size=384, background_id=None, max_tries=10):
    """Sample a square crop biased toward foreground pixels.

    Args:
        image: RGB uint8 array (H, W, 3).
        mask: Label map (H, W).
        crop_size: Side length of the square crop.
        background_id: Label treated as background; default uses ``BACKGROUND_ID``.
        max_tries: Attempts to find a crop with sufficient foreground.

    Returns:
        Tuple ``(crop_image, crop_mask)`` as uint8 arrays of shape (crop_size, crop_size, ...).

    Raises:
        None.
    """
    if background_id is None:
        background_id = BACKGROUND_ID
    h, w = mask.shape
    ch, cw = crop_size, crop_size

    if h < ch or w < cw:
        pad_h = max(0, ch - h)
        pad_w = max(0, cw - w)
        image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, borderType=cv2.BORDER_CONSTANT, value=(0, 0, 0))
        mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, borderType=cv2.BORDER_CONSTANT, value=background_id)
        h, w = mask.shape

    ys, xs = np.where(mask != background_id)

    if len(xs) > 0:
        for _ in range(max_tries):
            idx = np.random.randint(0, len(xs))
            cx, cy = xs[idx], ys[idx]

            x1 = np.clip(cx - cw // 2, 0, w - cw)
            y1 = np.clip(cy - ch // 2, 0, h - ch)

            crop_img = image[y1:y1+ch, x1:x1+cw]
            crop_mask = mask[y1:y1+ch, x1:x1+cw]

            if (crop_mask != background_id).mean() > 0.03:
                return crop_img, crop_mask

    x1 = np.random.randint(0, max(1, w - cw + 1))
    y1 = np.random.randint(0, max(1, h - ch + 1))
    return image[y1:y1+ch, x1:x1+cw], mask[y1:y1+ch, x1:x1+cw]
